## OLCI Sargassum Detection
`Train and evaluate a logistic regression on the feature database`

### 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# ignore warnings
import warnings
warnings.filterwarnings('ignore')

### 2. Load the feature database and define features

In [2]:
db_path    = 'sargassum_feature_db.csv'
model_path = 'sargassum_logreg.joblib'

db = pd.read_csv(db_path)

# Oa09, Oa12, Oa16 radiances plus the two indices
feature_cols = ['SGI', 'AFAI']
X = db[feature_cols].values
y = db['is_sargassum'].values

print(f"{len(db)} pixels: {y.sum()} Sargassum, {(y == 0).sum()} other")

2460 pixels: 1112 Sargassum, 1348 other


### 3. Split 70/30 and train

In [3]:
idx_train, idx_test = train_test_split(np.arange(len(db)), test_size = 0.3, stratify = y, random_state = 42)

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter = 1000))
model.fit(X[idx_train], y[idx_train])
y_pred = model.predict(X[idx_test])

### 4. Evaluate on the test set

In [4]:
print(confusion_matrix(y[idx_test], y_pred))
print(classification_report(y[idx_test], y_pred, target_names = ['other', 'sargassum'], digits = 3))

[[404   0]
 [  0 334]]
              precision    recall  f1-score   support

       other      1.000     1.000     1.000       404
   sargassum      1.000     1.000     1.000       334

    accuracy                          1.000       738
   macro avg      1.000     1.000     1.000       738
weighted avg      1.000     1.000     1.000       738



In [5]:
# same test set, broken down by scene
test = db.iloc[idx_test].assign(pred = y_pred)
for product, grp in test.groupby('product'):
    print(product[16:31], len(grp), 'errors:', (grp['pred'] != grp['is_sargassum']).sum())

# misclassified test pixels
test[test['pred'] != test['is_sargassum']][['product', 'Name', 'label', 'SGI', 'AFAI']]

20180513T134359 125 errors: 0
20240606T125452 239 errors: 0
20260624T130230 131 errors: 0
20200522T132234 112 errors: 0
20250510T125327 131 errors: 0


,product,Name,label,SGI,AFAI


### 5. Inspect the coefficients

Standardised, so magnitudes are comparable across features.

In [6]:
coef = pd.Series(model[-1].coef_[0], index = feature_cols).sort_values(key = np.abs, ascending = False)
coef.round(3)

AFAI    7.469
SGI     5.553
dtype: float64

### 6. Refit on all pixels and save the model

In [7]:
model.fit(X, y)
joblib.dump({'model': model, 'feature_cols': feature_cols}, model_path)
print(f"Saved {model_path}")

Saved sargassum_logreg.joblib
